# Class 1

From `class1/`:

```bash
pip install -r requirements.txt
export PYTHONPATH=$PWD
docker compose up naive-server
```

Skip Part 1 and Part 6 if you are short on time. Run the cells in order.

## Part 0 — device check

Run the cell. Server should already be up: `docker compose up naive-server`

In [ ]:
import os, shutil, subprocess, sys

print("Python:", sys.version.split()[0])

if shutil.which("nvidia-smi"):
    print("nvidia-smi: OK — run the full lab here (Modal GPU or local NVIDIA)")
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
        check=False,
    )
elif sys.platform == "darwin":
    print("Mac local: no nvidia-smi — run Parts 0-7 on Modal GPU (see MODAL.md)")
else:
    print("No nvidia-smi — use Modal GPU or install NVIDIA drivers")

try:
    import torch
    print("Torch CUDA:", torch.cuda.is_available())
except ImportError:
    print("Torch not installed in this kernel")

## Part 1 — skip if short on time

Naive server must be up. Run the cell.

In [ ]:
import os
os.environ.setdefault("OPENAI_BASE_URL", "http://127.0.0.1:8000/v1")

!python scripts/part1_anatomy.py

## Part 2 — naive server

Read `naive_server/server.py`, then run the cell. Do not load-test yet.

In [ ]:
from lib.client_utils import make_client, timed_chat_completion, print_timing

client = make_client()
msgs = [{"role": "user", "content": "Say hello in five words."}]
r = timed_chat_completion(client, msgs, stream=True, max_tokens=16)
print_timing("smoke", r)

## Part 3 — break the server

Terminal A, leave running:

```bash
python scripts/part5_observability.py --url http://127.0.0.1:8000 --interval 1 --count 120
```

Terminal B: `watch -n1 nvidia-smi`. While A is still printing, run the cell below.

In [ ]:
!python scripts/part3_break_server.py

Each request keeps its own KV cache. The lock does not cap memory.

## Part 4 — RelayServe and llama.cpp

```bash
python scripts/download_gguf.py
docker compose --profile relay up
```

Then run the cell. The engine swap is what stops the OOM. The gateway does not.

In [ ]:
import os
os.environ["OPENAI_BASE_URL"] = "http://127.0.0.1:8080/v1"

!python scripts/part4_compare.py --concurrency 8

Engine fixes OOM. Gateway adds the queue. Check: `curl -s http://127.0.0.1:8080/metrics | jq`

## Part 5 — metrics

Run the cell while load is going.

In [ ]:
import asyncio, json
from lib.client_utils import fetch_metrics

for url in ["http://127.0.0.1:8000", "http://127.0.0.1:8080"]:
    try:
        m = asyncio.run(fetch_metrics(url + "/v1"))
        print(url, json.dumps(m, indent=2)[:800], "...\n")
    except Exception as e:
        print(url, "unavailable:", e)

## Part 6 — skip if short on time

```bash
docker compose --profile litellm up
python scripts/part6_litellm_demo.py
```

## Part 7 — four answers

What broke, what the engine fixed, what the gateway fixed, what you would alert on. Run the cell.

In [ ]:
answers = {
    "q1_what_broke": input("1) What broke first on the naive server and why? "),
    "q2_engine_vs_gateway": input("2) What did llama.cpp improve vs what did RelayServe improve? "),
    "q3_counterfactual": input("3) What if RelayServe routed to the naive server? "),
    "q4_alerts": input("4) Production alert metrics? "),
}
print("\n--- Save these for your submission ---")
for k, v in answers.items():
    print(f"{k}: {v}")